# 90 · Six uses of the continuous PT axis (from the literature review)

The literature review (`docs/research/continuous-analysis-literature.md`) ranked five ways to use a continuous
axis better than a segment-level analysis does. This notebook runs them on the elastic curve, one figure each.

**Protocol.** `docs/results/pt-continuous-axis-experiments.md` (frozen before any fit, commit 3f7d89d).

| Part | Question |
|---|---|
| A | Is each gene better described by steps at the segment labels or by a continuous shape (gradient, switch, bump)? |
| B | Where do pathways sit along the PT, and which pathways does a median curve flatten? |
| C | How much information about position do a pathway's genes carry, against the segment labels? |
| D | Human against mouse without levels, against within-species baselines |
| E | Where along the PT does the transcriptome change, against a discrete transcriptome? |
| F | Where along the PT does injury act, and what does a segment pseudobulk confuse? |

**What bounds the reading.** Two healthy and two IR male mice, one male human donor (two cortex sections).
Everything is descriptive. The coordinate's fold noise exceeds the within-segment spread in mouse S1 and S2 (C14).

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, '1')
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

PKG = results_root / 'package_pseudospace'
EXPORT = {'hvm': PKG / 'hvm' / 'pt_units.h5ad', 'aki': PKG / 'aki' / 'pt_units.h5ad'}
STAGE01 = PKG / 'runs' / 'hvm' / '01_units'
RUN_HVM = PKG / 'runs' / 'hvm'
upstream13 = results_root / 'minimal_pt_scfates'
T71 = results_root / 'pt_pathways_package_pseudospace'
T72 = results_root / 'pt_package_coordinate_validation'
T86 = results_root / 'pt_non_one_to_one_package'
output = results_root / 'pt_continuous_axis_experiments'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)

DS = 'human_vs_mouse'
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
SPECIES = {'mouse': MOUSE, 'human': HUMAN}
CONTROLS, IRS = ['Ctrl1A2', 'Ctrl1A4'], ['IR2A2', 'IR2A4']
AKI4 = CONTROLS + IRS
REQUIRED = [EXPORT['hvm'], EXPORT['aki'], *[STAGE01 / f'{s}_units.h5ad' for s in MOUSE + HUMAN],
            upstream13 / 'ortholog_map_used.csv', T72 / 'tables' / 'transitions.csv',
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json')],
            T86 / 'tables' / 'w_calls_mouse.csv',
            RUN_HVM / '03_integration' / 'pass1_labelled.h5ad', RUN_HVM / '03_integration' / 'pass2.h5ad']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70–72 and 86 first; see data/README.md)')

NOTEBOOK_LOGIC_VERSION = '90.continuous_axis.1'
FINGERPRINT = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '24'))
SEED = 90
# Protocol constants
DETECTION, USABLE = .02, .05
EFFECT, MARGIN = .25, 6.
MIN_SET = 10
N_SIM_GENES = 400                        # A: semi-synthetic genes
N_PERM_POS, N_PERM_AL = 2000, 500       # B: position and aligned-amplitude nulls
TOP_TIMELINE = 24
RANDOM_SIZES = (10, 15, 20, 30, 50, 75, 100, 150, 200, 300)
N_RANDOM_PER_SIZE, N_SHUFFLE = 200, 20  # C
N_END = 8                               # D
GRADED, FLATD = .5, .2
HALF_LC, N_PERM_LC = 2, 100              # E
WINDOW = .3
RESP, RESP_RELABEL = .5, .5              # F
SEGMENTS = ('S1', 'S2', 'S3')
LABELS = ('PT-S1', 'PT-S2', 'PT-S3')
LABEL_ORIENTATION = ['Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b']
RANSICK = {'Slc5a2': 'S1', 'Spp2': 'S1', 'Slc34a1': 'S1–S2', 'Cyp2e1': 'S2', 'Slc22a7': 'S3', 'Cyp7b1': 'S3'}

MM = 1 / 25.4
INK, LIGHT, GREY = '#0b0b0b', '#d9d8d4', '#9a9893'
SPEC_COLOR = {'Ctrl1A2': '#0072B2', 'Ctrl1A4': '#56B4E9', 'HUK1_COR1': '#D55E00', 'HUK1_MED1': '#E69F00',
              'IR2A2': '#B2182B', 'IR2A4': '#EF8A62'}
CLASS_COLOR = {'discrete': '#7f7f7f', 'discrete + within': '#b5a642', 'gradient': '#0072B2', 'switch': '#009E73',
               'bump': '#D55E00', 'trough': '#CC79A7', 'flat': LIGHT}
LABEL_COLOR = {'PT-S1': '#4C72B0', 'PT-S2': '#55A868', 'PT-S3': '#C44E52'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 200)
GUARDS, READING = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def panel(ax, letter):
    ax.text(-.02, 1.02, letter, transform=ax.transAxes, fontsize=8, fontweight='bold', ha='right', va='bottom')


def seg_lines(ax, cuts=(1, 2), vertical=True):
    for b in cuts:
        (ax.axvline if vertical else ax.axhline)(b, color=LIGHT, lw=.6, zorder=0)


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)

## 2 · Inputs and guards (not cached)

89's guards: G1 `hvm_inputs` reproduces 71; G2 72's transitions; G3 the stage-01 native panels match the export
unit by unit; G5 86's mouse NB fits cover the PT-expressed mouse genes; G7 the AKI control transitions are ordered.

In [ ]:
import anndata as ad
from scipy import sparse
from scipy.stats import spearmanr, gaussian_kde, fisher_exact, norm
from joblib import Parallel, delayed

from pseudospace import continuous_axis as ca
from pseudospace import positional_programs as pp
from pseudospace import species_calls as sc
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.package_inputs import hvm_inputs, read_pt_units, support_positions
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from statsmodels.stats.multitest import multipletests

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
UNITS = read_pt_units(EXPORT['hvm'], 'hvm')
PROV = UNITS.uns['provenance']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['hvm'], PROV['fingerprints']['pass1']
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES71 = list(INP['genes'])
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
assert GENES71 == universe71[universe71.eligible.astype(bool)].gene.astype(str).tolist(), 'G1: universe differs from 71'
SETS71 = INP['gene_sets']
assert SETS71 == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G1: gene sets differ from 71'
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = np.asarray(INP['segment']).astype(str)
LIBRARY = np.asarray(INP['library'], float)
OBS = INP['obs']
IDENT = OBS['primary_identity'].astype(str).to_numpy()
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(POSITION)
GUARDS.append(f'G1: {len(GENES71):,} genes, {len(SETS71):,} sets and {len(POSITION):,} units equal 71')

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, SEGMENT, SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)
MOUSE_CUTS = sc.mean_transitions(TRANSITIONS, MOUSE)
UNITS_MOUSE = sc.to_segment_units(POSITION, MOUSE_CUTS)
GUARDS.append(f'G2: transitions reproduce 72\'s; mouse cuts {MOUSE_CUTS[0]:.3f}, {MOUSE_CUTS[1]:.3f}')

NATIVE, PANEL = {}, {}
for species, names in SPECIES.items():
    blocks = []
    for name in names:
        native = ad.read_h5ad(STAGE01 / f'{name}_units.h5ad')
        rows = np.flatnonzero(SPECIMEN == name)
        ids = OBS.polygon_id.astype(str).to_numpy()[rows]
        assert pd.Index(ids).isin(native.obs_names).all() and pd.Index(ids).is_unique, f'G3: {name} units missing'
        x = sparse.csr_matrix(native[list(ids)].X, dtype=float)
        export_x = sparse.csr_matrix(UNITS[OBS.index[rows]].X, dtype=float)
        symbols = UNITS.var_names if species == 'mouse' else pd.Index(UNITS.var.human_symbol.astype(str))
        cols = native.var_names.get_indexer(symbols)
        assert (cols >= 0).all() and (x[:, cols] != export_x).nnz == 0, f'G3: {name} counts differ from the export'
        PANEL[species] = pd.Index(native.var_names.astype(str))
        blocks.append((rows, x))
    NATIVE[species] = {'rows': np.concatenate([r for r, _ in blocks]),
                       'counts': sparse.vstack([x for _, x in blocks]).tocsr()}
GUARDS.append('G3: every export unit is in its stage-01 file; shared genes identical')

INTEGRATION = set()
for name in ('pass1_labelled.h5ad', 'pass2.h5ad'):
    obj = ad.read_h5ad(RUN_HVM / '03_integration' / name, backed='r')
    INTEGRATION |= set(obj.var_names[obj.var['highly_variable_for_harmony'].to_numpy(bool)].astype(str))
    obj.file.close()
BUILT = INTEGRATION | set(LABEL_ORIENTATION)
TO_HUMAN = pd.Series(UNITS.var.human_symbol.astype(str).to_numpy(), index=UNITS.var_names.astype(str))
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s; built genes {len(BUILT):,}')

## 3 · Binned profiles (89's)

U_PT units; fixed 0.1 bins of mouse units on [0, 2.5); per mouse, each gene's log2 rate per bin minus its mean,
minus the reference (the median usable gene per bin). Human: 25 equal bins over each section's 1–99% range.

In [ ]:
U_PT = IDENT == 'Proximal_Tubule'


def species_rows(species):
    rows = NATIVE[species]['rows']
    keep = U_PT[rows]
    return rows[keep], NATIVE[species]['counts'][keep]


def detection_by(counts, specimen, names):
    return np.array([np.asarray((counts[specimen == n] > 0).mean(axis=0)).ravel() for n in names])


H = {}
for species, names in SPECIES.items():
    rows, x = species_rows(species)
    spec = SPECIMEN[rows]
    det = detection_by(x, spec, names)
    not_mt = ~PANEL[species].str.upper().str.startswith('MT-')
    expressed = (det >= DETECTION).all(axis=0) & not_mt
    genes = np.flatnonzero(expressed)
    H[species] = {'rows': rows, 'counts': x[:, genes], 'genes': list(PANEL[species][genes]), 'specimen': spec,
                  'library': LIBRARY[rows], 'segment': SEGMENT[rows], 'position': POSITION[rows],
                  'usable': (det[:, genes] >= USABLE).all(axis=0)}
    H[species]['index'] = pd.Series(np.arange(len(genes)), index=H[species]['genes'])

W86 = stored(T86, 'w_mouse_pt_shared')
W86_GENES = pd.read_csv(T86 / 'tables' / 'w_calls_mouse.csv', usecols=['gene']).gene.astype(str).tolist()
assert len(W86['alpha']) == len(H['mouse']['genes']) and W86_GENES == H['mouse']['genes'], \
    'G5: 86\'s mouse W run covers a different gene set'
ALPHA = np.asarray(W86['alpha'], float)
GUARDS.append(f"G5: 86's cached NB fits cover the {len(H['mouse']['genes']):,} PT-expressed mouse genes")

m = H['mouse']
m['units'] = UNITS_MOUSE[m['rows']]
BIN, CENTRES = pp.fixed_bins(m['units'])
NB = len(CENTRES)
m['bin'] = BIN
m['profiles'] = pp.bin_profiles(m['counts'], m['library'], BIN, NB, m['specimen'], MOUSE)
m['reference'] = {n: pp.reference_profile(m['profiles'][n], m['usable']) for n in MOUSE}
assert np.all([np.isfinite(m['reference'][n]) for n in MOUSE]), 'bins without enough units in a mouse'
m['deviation'] = {n: m['profiles'][n] - m['reference'][n] for n in MOUSE}

hh = H['human']
hh['bin'] = np.full(len(hh['rows']), -1)
for n in HUMAN:
    r = hh['specimen'] == n
    lo, hi = np.quantile(hh['position'][r], [.01, .99])
    e = np.linspace(lo, hi, 26)
    b = np.clip(np.searchsorted(e, hh['position'][r], side='right') - 1, 0, 24)
    b[(hh['position'][r] < lo) | (hh['position'][r] > hi)] = -1
    hh['bin'][r] = b
hh['profiles'] = {}
for n in HUMAN:
    r = hh['specimen'] == n
    hh['profiles'].update(pp.bin_profiles(hh['counts'][r], hh['library'][r], hh['bin'][r], 25, hh['specimen'][r], [n]))
hh['reference'] = {n: pp.reference_profile(hh['profiles'][n], hh['usable']) for n in HUMAN}
hh['deviation'] = {n: hh['profiles'][n] - hh['reference'][n] for n in HUMAN}

U_IDX = np.flatnonzero(m['usable'])
U_GENES = np.array(m['genes'])[U_IDX]
POS_IN_USABLE = pd.Series(np.arange(len(U_IDX)), index=U_IDX)
FP = np.array([g not in BUILT for g in U_GENES])
DM = {n: m['deviation'][n][U_IDX] for n in MOUSE}
RATE = {n: np.asarray(m['counts'][m['specimen'] == n].sum(axis=0)).ravel() / m['library'][m['specimen'] == n].sum()
        for n in MOUSE}
LOG_CP = np.log2(np.mean([RATE[n] * 1e4 for n in MOUSE], axis=0) + 1e-6)
STRATA = pd.qcut(LOG_CP[U_IDX], 10, labels=False, duplicates='drop')
WEIGHT = {n: np.bincount(BIN[(m['specimen'] == n) & (BIN >= 0)], minlength=NB).astype(float) for n in MOUSE}
FRAC = {n: ca.label_fractions(BIN[m['specimen'] == n], m['segment'][m['specimen'] == n], NB, LABELS) for n in MOUSE}
assert all(np.isfinite(FRAC[n]).all() for n in MOUSE)
SET_IDS = [s for s in SETS71]


def usable_members(genes, protected=False):
    """Row positions in the usable-gene arrays of a set's members (fold-protected only when ``protected``)."""
    idx = m['index'].reindex(list(dict.fromkeys(genes))).dropna().astype(int).to_numpy()
    idx = idx[m['usable'][idx]]
    pos = POS_IN_USABLE[idx].to_numpy()
    return pos[FP[pos]] if protected else pos


SET_U = {s: usable_members(SETS71[s]) for s in SET_IDS}
SET_IDS = [s for s in SET_IDS if len(SET_U[s]) >= MIN_SET]
SET_FP = {s: usable_members(SETS71[s], protected=True) for s in SETS71}
print({sp: (len(H[sp]['genes']), int(H[sp]['usable'].sum())) for sp in SPECIES}, 'PT-expressed, usable genes;',
      f'{len(U_IDX):,} usable mouse genes, {int(FP.sum()):,} fold-protected; {len(SET_IDS):,} sets ≥ {MIN_SET}')

code_key = digest([NOTEBOOK_LOGIC_VERSION, Path(ca.__file__).read_text(), Path(pp.__file__).read_text()])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None):
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def cached_table(stage, compute, *, params=None, inputs=None):
    return cached_frame(stage, compute, root=cache, params=params, inputs=inputs, code=code_key)


def short(s, n=46):
    name = s.split('::', 1)[-1]
    for prefix in ('REACTOME_', 'KEGG_', 'HALLMARK_', 'WP_', 'GOBP_'):
        if name.startswith(prefix):
            name = name[len(prefix):]
    name = name.replace('_', ' ').strip()
    name = name[:1].upper() + name[1:].lower() if name.isupper() else name
    return name if len(name) <= n else name[:n - 1] + '…'


PROFILE_INPUTS = {'deviation': [DM[n] for n in MOUSE], 'weights': [WEIGHT[n] for n in MOUSE],
                  'frac': [FRAC[n] for n in MOUSE]}

## 4 · A · Gene shape classes, with segment steps as a competing model

Six models per gene on the two mice's deviation profiles (weights = units per bin): flat, linear gradient,
logistic switch, Gaussian bump, discrete (constant within each label, with each bin's label shares), discrete plus
a slope within each label. Lowest BIC; flat when the fitted range is < 0.25 log2. ΔBIC = BIC(D) − best continuous.

In [ ]:
def ladder_compute():
    joint = ca.fit_ladder([DM[n] for n in MOUSE], [WEIGHT[n] for n in MOUSE], CENTRES, [FRAC[n] for n in MOUSE])
    joint.insert(0, 'gene', U_GENES)
    return joint


def single_compute(n):
    def run():
        t = ca.fit_ladder([DM[n]], [WEIGHT[n]], CENTRES, [FRAC[n]])
        t.insert(0, 'gene', U_GENES)
        return t
    return run


LADDER = cached_table('ladder_joint', ladder_compute, inputs=PROFILE_INPUTS)
LADDER_SINGLE = {n: cached_table(f'ladder_{n}', single_compute(n), inputs={'d': DM[n], 'w': WEIGHT[n], 'f': FRAC[n]})
                 for n in MOUSE}
CLS = ca.ladder_class(LADDER.drop(columns='gene'))
CLS.insert(0, 'gene', U_GENES)
CLS['fold-protected'] = FP
CLS['log2 CP10k'] = LOG_CP[U_IDX]
CLS_SINGLE = {n: ca.ladder_class(LADDER_SINGLE[n].drop(columns='gene')) for n in MOUSE}
for n in MOUSE:
    CLS[f'class {n}'] = CLS_SINGLE[n]['class'].to_numpy()


def change_position(row):
    """A's change position of a gene: switch midpoint, bump peak, else the half-crossing of its fitted D or L curve."""
    if row['class'] == 'switch':
        return row['S position']
    if row['class'] in ('bump', 'trough'):
        return row['B position']
    if row['class'] in ('discrete', 'discrete + within'):
        return ca.half_point(CENTRES, ca.ladder_curve('D', row, CENTRES, (FRAC[MOUSE[0]] + FRAC[MOUSE[1]]) / 2))
    if row['class'] == 'gradient':
        return ca.half_point(CENTRES, ca.ladder_curve('L', row, CENTRES))
    return np.nan


CLS['change position'] = [change_position(r) for _, r in CLS.iterrows()]
CLS.to_csv(tables / 'a_gene_classes.csv.gz', index=False)
CLASS_ORDER = ['discrete', 'discrete + within', 'gradient', 'switch', 'bump', 'trough']
nonflat = ~CLS['class'].eq('flat')
A_COUNTS = pd.DataFrame({
    'all usable': CLS['class'].value_counts(),
    'fold-protected': CLS.loc[FP, 'class'].value_counts(),
    'replicated (both mice same class)': CLS.loc[nonflat & CLS[f'class {MOUSE[0]}'].eq(CLS['class'])
                                                 & CLS[f'class {MOUSE[1]}'].eq(CLS['class']), 'class'].value_counts(),
}).reindex(CLASS_ORDER + ['flat']).fillna(0).astype(int)
display(A_COUNTS)
agree = (CLS.loc[nonflat, f'class {MOUSE[0]}'] == CLS.loc[nonflat, f'class {MOUSE[1]}']).mean()
pa = CLS.loc[nonflat, f'class {MOUSE[0]}'].value_counts(normalize=True)
pb = CLS.loc[nonflat, f'class {MOUSE[1]}'].value_counts(normalize=True)
agree_chance = float((pa * pb.reindex(pa.index).fillna(0)).sum())
A_CD = pd.DataFrame({'all usable': CLS.loc[nonflat, 'continuous vs discrete'].value_counts(),
                     'fold-protected': CLS.loc[nonflat & FP, 'continuous vs discrete'].value_counts()}).fillna(0).astype(int)
display(A_CD)
print(f'A replication: same class in both mice {agree:.3f} of non-flat genes (chance {agree_chance:.3f})')
LOC = CLS[nonflat & CLS['class'].isin(['switch', 'bump', 'trough'])].groupby(['class', 'location']).size().unstack(fill_value=0)
display(LOC)
display(CLS[CLS.gene.isin(RANSICK)][['gene', 'class', 'location', 'switch width', 'dBIC continuous', 'fold-protected',
                                     f'class {MOUSE[0]}', f'class {MOUSE[1]}']])

### A · Semi-synthetic calibration

400 non-flat fold-protected genes: NB counts on the real units with each gene's 86 dispersion, under (i) its fitted
D model (each unit takes its label's level) and (ii) its best continuous model at the unit's position.

In [ ]:
rng_sim = np.random.default_rng(SEED)
SIM_GENES = np.sort(rng_sim.choice(np.flatnonzero(nonflat.to_numpy() & FP), size=N_SIM_GENES, replace=False))
SEG_CODE = {lab: k for k, lab in enumerate(LABELS)}


def simulate_truth(kind, seed):
    rng = np.random.default_rng(seed)
    prof = {}
    for n in MOUSE:
        r = m['specimen'] == n
        units, lib = m['units'][r], m['library'][r]
        seg = np.array([SEG_CODE.get(s, -1) for s in m['segment'][r]])
        shape = m['reference'][n][np.clip(BIN[r], 0, NB - 1)] * np.log(2)
        effects = []
        for g in SIM_GENES:
            row = CLS.loc[g]
            if kind == 'discrete':
                levels = np.array([0., row['D S2-S1'], row['D S3-S1']])
                eff = np.where(seg >= 0, levels[np.clip(seg, 0, 2)], 0.)
            else:
                eff = ca.ladder_curve(row['continuous model'], row, units)
            effects.append(eff * np.log(2))
        gi = U_IDX[SIM_GENES]
        counts = pp.simulate_counts(lib, RATE[n][gi], shape, np.array(effects), ALPHA[gi], rng)
        y = pp.bin_profiles(sparse.csr_matrix(counts), lib, BIN[r], NB, np.array([n] * r.sum()), [n])[n]
        prof[n] = y - m['reference'][n]
    t = ca.fit_ladder([prof[n] for n in MOUSE], [WEIGHT[n] for n in MOUSE], CENTRES, [FRAC[n] for n in MOUSE])
    return t


SIM = {}
for k, kind in enumerate(('discrete', 'continuous')):
    SIM[kind] = ca.ladder_class(cached_table(f'sim_{kind}', lambda kind=kind, k=k: simulate_truth(kind, SEED + 10 + k),
                                             params={'n': N_SIM_GENES, 'seed': SEED}, inputs={**PROFILE_INPUTS,
                                             'genes': SIM_GENES, 'alpha': ALPHA, 'cls': CLS[['D S2-S1', 'D S3-S1']].to_numpy()}))
A_SIM = pd.DataFrame({kind: SIM[kind]['continuous vs discrete'].value_counts(normalize=True) for kind in SIM}).fillna(0)
display(A_SIM.round(3))
real_cont = float((CLS.loc[nonflat & FP, 'continuous vs discrete'] == 'continuous').mean())
false_cont = float((SIM['discrete'].loc[~SIM['discrete']['class'].eq('flat'), 'continuous vs discrete'] == 'continuous').mean())
false_disc = float((SIM['continuous'].loc[~SIM['continuous']['class'].eq('flat'), 'continuous vs discrete'] == 'discrete').mean())
print(f'A: continuous favoured in {real_cont:.3f} of non-flat fold-protected genes; D-truth called continuous '
      f'{false_cont:.3f}; continuous-truth called discrete {false_disc:.3f}')

# bumps replicated: joint B, both single fits B of the same sign, peaks within 0.3
bump_rep = CLS[CLS['class'].isin(['bump', 'trough'])].copy()
for n in MOUSE:
    bump_rep[f'peak {n}'] = CLS_SINGLE[n].loc[bump_rep.index, 'B position']
bump_rep['replicated'] = (bump_rep[f'class {MOUSE[0]}'].eq(bump_rep['class']) & bump_rep[f'class {MOUSE[1]}'].eq(bump_rep['class'])
                          & ((bump_rep[f'peak {MOUSE[0]}'] - bump_rep[f'peak {MOUSE[1]}']).abs() <= .3 + 1e-9))
bump_rep.to_csv(tables / 'a_bumps.csv', index=False)
print(f"A bumps/troughs: {len(bump_rep)}, replicated {int(bump_rep.replicated.sum())} "
      f"(fold-protected {int((bump_rep.replicated & bump_rep['fold-protected']).sum())})")
display(bump_rep[bump_rep.replicated].sort_values('range best', ascending=False)
        [['gene', 'class', 'B position', 'B width', 'range best', 'location', 'fold-protected', 'log2 CP10k']].head(30))

### Figure 90a

In [ ]:
def exemplar(cls_name):
    c = CLS[CLS['class'].eq(cls_name) & CLS[f'class {MOUSE[0]}'].eq(cls_name) & CLS[f'class {MOUSE[1]}'].eq(cls_name)]
    if cls_name in ('bump', 'trough'):
        c = c[c.index.isin(bump_rep.index[bump_rep.replicated])]
    if not len(c):
        c = CLS[CLS['class'].eq(cls_name)]
    if (c['log2 CP10k'] >= 0).sum() >= 3:
        c = c[c['log2 CP10k'] >= 0]
    margin = c[[f'bic {mm}' for mm in ca.MODELS]].apply(lambda r: np.sort(r.to_numpy())[1] - r.min(), axis=1)
    score = margin.rank() + c['range best'].rank() + c['fold-protected'].astype(int) * len(c)
    return int(score.idxmax())


fig = plt.figure(figsize=(180 * MM, 132 * MM))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.3], width_ratios=[1, 1, 1, 1.05], hspace=.55, wspace=.75)
REPLICATED = nonflat & CLS[f'class {MOUSE[0]}'].eq(CLS['class']) & CLS[f'class {MOUSE[1]}'].eq(CLS['class'])
ax = fig.add_subplot(gs[0, 0])
cnt = A_COUNTS.loc[CLASS_ORDER]
y = np.arange(len(CLASS_ORDER))[::-1]
ax.barh(y, cnt['all usable'], color=[CLASS_COLOR[c] for c in CLASS_ORDER], alpha=.35, height=.75)
ax.barh(y, cnt['replicated (both mice same class)'], color=[CLASS_COLOR[c] for c in CLASS_ORDER], height=.75)
ax.set_yticks(y, CLASS_ORDER)
ax.set_xlabel('genes')
panel(ax, 'a')

ax = fig.add_subplot(gs[0, 1])
READ_ORDER = ['continuous', 'undecided', 'discrete']
READ_COLOR = {'continuous': '#009E73', 'undecided': LIGHT, 'discrete': GREY}
cols_b = {'data': CLS.loc[nonflat & FP, 'continuous vs discrete'],
          'step truth': SIM['discrete'].loc[~SIM['discrete']['class'].eq('flat'), 'continuous vs discrete'],
          'continuous truth': SIM['continuous'].loc[~SIM['continuous']['class'].eq('flat'), 'continuous vs discrete']}
bottom = np.zeros(3)
for rd in READ_ORDER:
    v = np.array([(c == rd).mean() for c in cols_b.values()])
    ax.bar(np.arange(3), v, bottom=bottom, color=READ_COLOR[rd], width=.7, label=rd)
    bottom += v
ax.set_xticks(np.arange(3), list(cols_b), rotation=25, ha='right')
ax.set_ylabel('share of non-flat genes')
ax.set_ylim(0, 1)
ax.legend(fontsize=5, loc='lower center', bbox_to_anchor=(.5, 1.07), ncol=3, handlelength=1, columnspacing=.6)
panel(ax, 'b')

ax = fig.add_subplot(gs[0, 2])
sw = CLS[REPLICATED & CLS['class'].isin(['switch', 'bump', 'trough'])]
for cname in ('switch', 'bump', 'trough'):
    v = sw.loc[sw['class'].eq(cname), 'change position']
    ax.hist(v, bins=np.arange(0, 2.55, .1), color=CLASS_COLOR[cname], lw=1.1, label=cname, histtype='step')
seg_lines(ax)
ax.set_xlabel('mouse segment units')
ax.set_ylabel('replicated genes')
ax.legend(fontsize=5.5)
panel(ax, 'c')

ax = fig.add_subplot(gs[0, 3])
nf = CLS[REPLICATED]
ax.scatter(nf['DS jump'], nf['DS within slope'], s=2.5, c=[CLASS_COLOR[c] for c in nf['class']], lw=0, alpha=.7,
           rasterized=True)
ax.set_xlabel('jump at boundaries (log2)')
ax.set_ylabel('slope within segments (log2 / unit)')
ax.set_xlim(0, np.quantile(nf['DS jump'], .995))
ax.set_ylim(0, np.quantile(nf['DS within slope'], .995))
panel(ax, 'd')

ax = fig.add_subplot(gs[1, :2])
MEAN_DEV = (DM[MOUSE[0]] + DM[MOUSE[1]]) / 2
order, bounds = [], []
for cname in CLASS_ORDER:
    idx = CLS.index[CLS['class'].eq(cname) & REPLICATED]
    pos = CLS.loc[idx, 'change position'].fillna(1.25).to_numpy()
    sign = np.sign(MEAN_DEV[idx, -5:].mean(axis=1) - MEAN_DEV[idx, :5].mean(axis=1))
    if cname in ('bump', 'trough'):
        sign = np.ones(len(idx))
    o = idx[np.lexsort((pos, -sign))]
    order.extend(o)
    bounds.append(len(order))
im = ax.imshow(np.clip(MEAN_DEV[order], -1.5, 1.5), aspect='auto', cmap='RdBu_r', norm=TwoSlopeNorm(0, -1.5, 1.5),
               extent=(0, 2.5, len(order), 0), interpolation='nearest', rasterized=True)
for b in bounds[:-1]:
    ax.axhline(b, color=INK, lw=.5)
seg_lines(ax)
ax.set_yticks([(a + b) / 2 for a, b in zip([0] + bounds[:-1], bounds)], CLASS_ORDER)
ax.tick_params(axis='y', length=0)
ax.set_xlabel('mouse segment units')
cb = fig.colorbar(im, ax=ax, fraction=.025, pad=.01)
cb.set_label('log2, centred', labelpad=1)
panel(ax, 'e')

sub = gs[1, 2:].subgridspec(2, 3, hspace=.6, wspace=.5)
fm = (FRAC[MOUSE[0]] + FRAC[MOUSE[1]]) / 2
for k, cname in enumerate(CLASS_ORDER):
    ax = fig.add_subplot(sub[k // 3, k % 3])
    g = exemplar(cname)
    row = CLS.loc[g]
    for n in MOUSE:
        ax.plot(CENTRES, DM[n][g] - np.average(DM[n][g], weights=WEIGHT[n]), 'o', ms=1.6, color=SPEC_COLOR[n], mew=0)
    best = row['best'] if row['best'] in ('L', 'S', 'B', 'D') else 'D'
    for mdl, ls, col in ((best, '-', CLASS_COLOR[cname]), ('D', '--', INK)):
        curve = ca.ladder_curve(mdl, row, CENTRES, fm)
        ax.plot(CENTRES, curve - np.average(curve, weights=WEIGHT[MOUSE[0]] + WEIGHT[MOUSE[1]]), ls, color=col, lw=.9)
    seg_lines(ax)
    ax.set_title(row['gene'], fontsize=6, style='italic', fontweight='normal')
    ax.tick_params(labelsize=5)
    if k == 0:
        panel(ax, 'f')
        ax.legend(handles=[plt.Line2D([], [], marker='o', ls='', ms=2, color=SPEC_COLOR[n], label=n) for n in MOUSE]
                  + [plt.Line2D([], [], color=INK, ls='--', lw=.8, label='steps')], fontsize=4.5, loc='upper left',
                  handlelength=1.2, borderpad=.2)
save(fig, 'fig90a_shape_classes')

## 5 · B · Where pathways sit, and pathways a median curve flattens

In [ ]:
COM = {n: ca.centre_of_mass(DM[n], CENTRES) for n in MOUSE}
rng_b = np.random.default_rng(SEED + 1)
PERMS = pp.strata_permutations(STRATA, N_PERM_POS, rng_b)


def set_position_test(sets_u):
    rows = []
    for s, u in sets_u.items():
        out = {'set': s, 'members': len(u)}
        for n in MOUSE:
            obs = COM[n][u].mean()
            null = COM[n][PERMS[:, u]].mean(axis=1)
            mu, sd = null.mean(), null.std(ddof=1)
            out[f'CoM {n}'] = obs
            out[f'z {n}'] = (obs - mu) / sd
            out[f'p empirical {n}'] = (1 + np.sum(np.abs(null - mu) >= abs(obs - mu))) / (len(null) + 1)
            out[f'p {n}'] = 2 * norm.sf(abs(out[f'z {n}']))
        rows.append(out)
    t = pd.DataFrame(rows)
    same = np.sign(t[f'z {MOUSE[0]}']) == np.sign(t[f'z {MOUSE[1]}'])
    t['p'] = np.where(same, t[[f'p {n}' for n in MOUSE]].max(axis=1), 1.)
    t['q'] = multipletests(t['p'], method='fdr_bh')[1]
    t['z mean'] = t[[f'z {n}' for n in MOUSE]].mean(axis=1)
    t['direction'] = np.where(t['z mean'] > 0, 'late', 'early')
    t['positioned'] = t['q'] <= .05
    t['name'] = [short(s) for s in t['set']]
    return t


READING.append('B: p per mouse from the permutation z (normal), not the empirical p: 2,000 permutations floor the '
               'empirical p at 5e-4, which BH over about 1,400 sets cannot pass (seen in the first run: 0 fold-protected '
               'calls). The empirical p is kept as a column.')
B_POS = cached_table('b_set_positions', lambda: set_position_test({s: SET_U[s] for s in SET_IDS}),
                     params={'n_perm': N_PERM_POS, 'seed': SEED, 'p': 'normal from z'}, inputs={'com': [COM[n] for n in MOUSE], 'strata': STRATA})
FP_IDS = [s for s in SETS71 if len(SET_FP[s]) >= MIN_SET]
B_POS_FP = cached_table('b_set_positions_fp', lambda: set_position_test({s: SET_FP[s] for s in FP_IDS}),
                        params={'n_perm': N_PERM_POS, 'seed': SEED, 'p': 'normal from z'}, inputs={'com': [COM[n] for n in MOUSE], 'strata': STRATA})
print(f"B: {int(B_POS.positioned.sum())} of {len(B_POS)} sets positioned (early {int((B_POS.positioned & B_POS.direction.eq('early')).sum())}, "
      f"late {int((B_POS.positioned & B_POS.direction.eq('late')).sum())}); fold-protected {int(B_POS_FP.positioned.sum())} of {len(B_POS_FP)}")
B_POS.to_csv(tables / 'b_set_positions.csv', index=False)
B_POS_FP.to_csv(tables / 'b_set_positions_fold_protected.csv', index=False)


def aligned_one(u, d0, d1, perms):
    """A_med, cross-validated A_al and its null for one set's member rows ``u``."""
    med = np.mean([np.ptp(np.nanmedian(d[u], axis=0)) for d in (d0, d1)])
    a1, s1 = ca.aligned_amplitude(d0[u], d1[u])
    a2, s2 = ca.aligned_amplitude(d1[u], d0[u])
    null = np.array([(ca.aligned_amplitude(d0[p[u]], d1[p[u]])[0] + ca.aligned_amplitude(d1[p[u]], d0[p[u]])[0]) / 2
                     for p in perms])
    agree_sign = s1 == s2
    return {'members': len(u), 'A_med': med, 'A_al': (a1 + a2) / 2, 'A_al null 95': np.quantile(null, .95),
            'A_al percentile': float((null < (a1 + a2) / 2).mean()),
            'sign + (agreed)': int(((s1 > 0) & agree_sign).sum()), 'sign - (agreed)': int(((s1 < 0) & agree_sign).sum()),
            'signs A': ''.join('+' if v > 0 else '-' for v in s1), 'signs B': ''.join('+' if v > 0 else '-' for v in s2)}


def aligned_rows(ids, members_of_set, perms):
    rows = Parallel(n_jobs=N_JOBS)(delayed(aligned_one)(members_of_set[s], DM[MOUSE[0]], DM[MOUSE[1]], perms) for s in ids)
    t = pd.DataFrame(rows)
    t.insert(0, 'set', list(ids))
    t['hidden by the median'] = ((t['A_al'] > t['A_al null 95']) & (t['A_al'] >= EFFECT) & (t['A_med'] < EFFECT)
                                 & (t['sign + (agreed)'] >= 3) & (t['sign - (agreed)'] >= 3))
    t['name'] = [short(s) for s in t['set']]
    return t


B_AL = cached_table('b_aligned', lambda: aligned_rows(SET_IDS, SET_U, PERMS[:N_PERM_AL]),
                    params={'n_perm': N_PERM_AL, 'seed': SEED}, inputs={'d': [DM[n] for n in MOUSE], 'strata': STRATA})
B_AL_FP = cached_table('b_aligned_fp', lambda: aligned_rows(FP_IDS, SET_FP, PERMS[:N_PERM_AL]),
                       params={'n_perm': N_PERM_AL, 'seed': SEED}, inputs={'d': [DM[n] for n in MOUSE], 'strata': STRATA})
B_AL.to_csv(tables / 'b_aligned_amplitude.csv', index=False)
B_AL_FP.to_csv(tables / 'b_aligned_amplitude_fold_protected.csv', index=False)
print(f"B: hidden by the median {int(B_AL['hidden by the median'].sum())} of {len(B_AL)} sets "
      f"(fold-protected {int(B_AL_FP['hidden by the median'].sum())} of {len(B_AL_FP)}); "
      f"A_al above null 95th: {int((B_AL['A_al'] > B_AL['A_al null 95']).sum())}")
display(B_AL[B_AL['hidden by the median']].sort_values('A_al', ascending=False)
        [['name', 'members', 'A_med', 'A_al', 'A_al null 95', 'sign + (agreed)', 'sign - (agreed)']].head(30))

READING.append('B timeline: gradient members are left out; the half-crossing of a straight line is always the axis '
               'middle, so including them piles every set at 1.25 (seen in the first run).')
flat_med = ((B_AL['A_med'] < EFFECT) & (B_AL['sign + (agreed)'] >= 3) & (B_AL['sign - (agreed)'] >= 3))
HIDDEN_EXPECTED = .05 * int(flat_med.sum())
print(f"B: hidden by the median {int(B_AL['hidden by the median'].sum())} against about {HIDDEN_EXPECTED:.0f} expected by "
      f"chance (5% of the {int(flat_med.sum())} sets with a flat median and both signs); A_al above null 95th in "
      f"{(B_AL['A_al'] > B_AL['A_al null 95']).mean():.3f} of sets")
pos_sets = B_POS[B_POS.positioned]
groups = pp.redundancy_groups({s: [U_GENES[i] for i in SET_U[s]] for s in pos_sets['set']})
REP = []
for g, ss in groups.groupby(groups).groups.items():
    REP.append(sorted(ss, key=lambda s: (-len(SET_U[s]), s))[0])
TOP = B_POS.set_index('set').loc[REP].assign(absz=lambda d: d['z mean'].abs()).sort_values('absz', ascending=False).head(TOP_TIMELINE)
display(TOP[['name', 'members', 'z mean', 'q']].round(4))

### Figure 90b

In [ ]:
def member_changes(s):
    """Change positions and directions (+1 on, −1 off) of a set's switch, bump, trough and discrete members."""
    rows = CLS.loc[SET_U[s]]
    rows = rows[~rows['class'].isin(['flat', 'gradient'])].dropna(subset=['change position'])
    direction = np.where(rows['class'].eq('switch'), np.sign(rows['S amp']),
                         np.where(rows['class'].eq('bump'), 1., np.where(rows['class'].eq('trough'), -1.,
                                  np.sign(np.where(rows['D S3-S1'] != 0, rows['D S3-S1'], rows['D S2-S1'])))))
    return rows['change position'].to_numpy(), direction


fig = plt.figure(figsize=(180 * MM, 118 * MM))
gs = fig.add_gridspec(2, 3, width_ratios=[1.5, 1, 1], hspace=.5, wspace=.5)
ax = fig.add_subplot(gs[:, 0])
tl = [s for s in TOP.index if len(member_changes(s)[0]) >= 4]
tl.sort(key=lambda s: B_POS.set_index('set').loc[s, 'z mean'])
grid = np.linspace(0, 2.5, 251)
for k, s in enumerate(tl):
    pos, dirn = member_changes(s)
    for sg, col in ((1, '#C44E52'), (-1, '#4C72B0')):
        p_ = pos[dirn == sg]
        if len(p_) >= 2:
            dens = gaussian_kde(p_, bw_method=.4)(grid) * len(p_) / len(pos)
            ax.fill_between(grid, k, k + dens / 1.6, color=col, alpha=.45, lw=0)
            ax.plot(grid, k + dens / 1.6, color=col, lw=.5)
seg_lines(ax)
ax.set_yticks(np.arange(len(tl)) + .2, [TOP.loc[s, 'name'] for s in tl], fontsize=5)
for lab_, s in zip(ax.get_yticklabels(), tl):
    lab_.set_color('#C44E52' if TOP.loc[s, 'z mean'] > 0 else '#4C72B0')
ax.set_xlim(0, 2.5)
ax.set_ylim(-.2, len(tl) + .6)
ax.set_xlabel('mouse segment units')
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color='#C44E52', alpha=.45, label='switching on'),
                   plt.Rectangle((0, 0), 1, 1, color='#4C72B0', alpha=.45, label='switching off')], fontsize=5,
          loc='lower center', bbox_to_anchor=(.5, 1.0), ncol=2)
panel(ax, 'a')

ax = fig.add_subplot(gs[0, 1])
ax.scatter(B_POS[f'z {MOUSE[0]}'], B_POS[f'z {MOUSE[1]}'], s=2, color=GREY, lw=0, alpha=.5, rasterized=True)
pp_ = B_POS[B_POS.positioned]
ax.scatter(pp_[f'z {MOUSE[0]}'], pp_[f'z {MOUSE[1]}'], s=3, c=np.where(pp_['z mean'] > 0, '#C44E52', '#4C72B0'), lw=0)
ax.axhline(0, color=LIGHT, lw=.5)
ax.axvline(0, color=LIGHT, lw=.5)
ax.set_xlabel(f'position z, {MOUSE[0]}')
ax.set_ylabel(f'position z, {MOUSE[1]}')
panel(ax, 'b')

ax = fig.add_subplot(gs[0, 2])
ax.scatter(B_AL['A_med'], B_AL['A_al'], s=2, color=GREY, lw=0, alpha=.5, rasterized=True)
hid = B_AL[B_AL['hidden by the median']]
ax.scatter(hid['A_med'], hid['A_al'], s=4, color='#D55E00', lw=0)
lim = max(B_AL['A_al'].quantile(.995), B_AL['A_med'].quantile(.995))
ax.plot([0, lim], [0, lim], color=INK, lw=.4, ls=':')
ax.axvline(EFFECT, color=LIGHT, lw=.5)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel('median-curve amplitude (log2)')
ax.set_ylabel('aligned amplitude, cross-mouse (log2)')
panel(ax, 'c')

sub = gs[1, 1:].subgridspec(2, 2, hspace=.6, wspace=.35)
ex_sets = list(TOP[TOP['z mean'] < 0].index[:2]) + list(TOP[TOP['z mean'] > 0].index[:2])
for k, s in enumerate(ex_sets):
    ax = fig.add_subplot(sub[k // 2, k % 2])
    for n in MOUSE:
        ax.plot(CENTRES, np.nanmedian(DM[n][SET_U[s]], axis=0), color=SPEC_COLOR[n], lw=1)
    seg_lines(ax)
    ax.axhline(0, color=LIGHT, lw=.4)
    ax.set_title(short(s, 30), fontsize=5.5, fontweight='normal', color='#C44E52' if TOP.loc[s, 'z mean'] > 0 else '#4C72B0')
    ax.tick_params(labelsize=5)
    if k == 0:
        panel(ax, 'd')
        ax.legend(fontsize=4.5, labels=MOUSE, handlelength=1, loc='lower left', borderpad=.2)
save(fig, 'fig90b_pathway_positions')

## 6 · C · Positional information: decoding position from a pathway's genes

Train on one mouse, test on the other. NB likelihood of each bin from the training mouse's binned rates and 86's
dispersions; the training mouse's bin occupancy as prior; MAP bin. Information = MI(true bin; decoded bin), plug-in
minus shuffles. Fold-protected genes only.

In [ ]:
FP_ROWS = U_IDX[FP]                      # rows in the PT-expressed mouse arrays
DEC = {}
for n in MOUSE:
    r = (m['specimen'] == n) & (BIN >= 0)
    DEC[n] = {'counts': m['counts'][r][:, FP_ROWS].tocsr(), 'library': m['library'][r], 'bin': BIN[r],
              'units': m['units'][r], 'label': np.array([SEG_CODE.get(s, -1) for s in m['segment'][r]])}
    rates = np.exp2(pp.bin_profiles(m['counts'][r][:, FP_ROWS], m['library'][r], BIN[r], NB, m['specimen'][r], [n],
                                    centre=False, min_units=1)[n])
    DEC[n]['rates'] = rates
    occ = np.bincount(BIN[r], minlength=NB) / r.sum()
    DEC[n]['log_prior'] = np.log(np.maximum(occ, 1e-6))
    lab_bin = np.zeros((3, NB))
    for k in range(3):
        lab_bin[k] = np.bincount(BIN[r][DEC[n]['label'] == k], minlength=NB) + 1e-3
        lab_bin[k] /= lab_bin[k].sum()
    DEC[n]['log_p_bin_label'] = np.log(lab_bin)
    assert (DEC[n]['label'] >= 0).all()
FP_ALPHA = ALPHA[FP_ROWS]
DIRECTIONS = [(MOUSE[0], MOUSE[1]), (MOUSE[1], MOUSE[0])]


def decode(cols, train, test, *, with_labels=False, onoff=False):
    """Posterior over bins for the test mouse's units from the genes ``cols`` (columns of the fold-protected arrays)."""
    tr, te = DEC[train], DEC[test]
    if onoff:
        r = (m['specimen'] == train) & (BIN >= 0)
        det = np.vstack([np.asarray((tr['counts'][tr['bin'] == b][:, cols] > 0).mean(axis=0)).ravel() for b in range(NB)]).T
        det = np.clip(det, 1e-3, 1 - 1e-3)
        D = (te['counts'][:, cols] > 0).astype(float)
        ll = D @ np.log(det / (1 - det)) + np.log(1 - det).sum(axis=0)[None, :]
    elif len(cols):
        ll = ca.nb_bin_loglik(te['counts'][:, cols], te['library'], tr['rates'][cols], FP_ALPHA[cols])
    else:
        ll = np.zeros((len(te['bin']), NB))
    prior = tr['log_p_bin_label'][te['label']] if with_labels else tr['log_prior'][None, :]
    return ca.log_posterior(ll, prior)


def info(post, test, rng):
    te = DEC[test]
    mapb = post.argmax(axis=1)
    mean_pos = np.exp(post) @ CENTRES
    err = mean_pos - te['units']
    return {'I': ca.corrected_mi(te['bin'], mapb, NB, NB, rng, N_SHUFFLE),
            'I within': ca.within_label_mi(te['bin'], mapb, te['label'], NB, NB, rng, N_SHUFFLE),
            'median abs error': float(np.median(np.abs(err))),
            'rmse by bin': np.array([np.sqrt(np.mean(err[te['bin'] == b] ** 2)) for b in range(NB)])}


def info_both(cols, **kw):
    rng = np.random.default_rng(SEED + 7)
    res = [info(decode(cols, a, b, **kw), b, rng) for a, b in DIRECTIONS]
    return {k: np.mean([r[k] for r in res], axis=0) for k in res[0]}


def labels_only():
    rng = np.random.default_rng(SEED + 7)
    res = [info(decode(np.array([], int), a, b, with_labels=True), b, rng) for a, b in DIRECTIONS]
    return {k: np.mean([r[k] for r in res], axis=0) for k in res[0]}


FP_POS = pd.Series(np.arange(int(FP.sum())), index=np.flatnonzero(FP))   # usable position -> fold-protected column


def set_cols(s):
    return FP_POS[SET_FP[s]].to_numpy()


def c_compute():
    out = {}
    lab = labels_only()
    out.update({f'labels|{k}': v for k, v in lab.items()})
    allc = info_both(np.arange(int(FP.sum())))
    out.update({f'all|{k}': v for k, v in allc.items()})
    allcl = info_both(np.arange(int(FP.sum())), with_labels=True)
    out.update({f'all+labels|{k}': v for k, v in allcl.items()})
    res = Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(info_both)(set_cols(s)) for s in FP_IDS)
    for k in ('I', 'I within', 'median abs error'):
        out[f'sets|{k}'] = np.array([r[k] for r in res])
    out['sets|rmse by bin'] = np.vstack([r['rmse by bin'] for r in res])
    rng = np.random.default_rng(SEED + 3)
    strata_fp = STRATA[FP]
    rand_sets = []
    for size in RANDOM_SIZES:
        for _ in range(N_RANDOM_PER_SIZE):
            rand_sets.append((size, np.concatenate([rng.choice(np.flatnonzero(strata_fp == d), size=c, replace=False)
                                                    for d, c in zip(*np.unique(rng.choice(strata_fp, size=size, replace=True),
                                                                               return_counts=True))])))
    resr = Parallel(n_jobs=N_JOBS, prefer='threads')(delayed(info_both)(c) for _, c in rand_sets)
    out['random|size'] = np.array([s for s, _ in rand_sets])
    for k in ('I', 'I within', 'median abs error'):
        out[f'random|{k}'] = np.array([r[k] for r in resr])
    return out


C_RES = cached('c_decoding', c_compute, params={'sizes': RANDOM_SIZES, 'n': N_RANDOM_PER_SIZE, 'shuffle': N_SHUFFLE,
                                                'seed': SEED},
               inputs={'counts': [digest(DEC[n]['counts']) for n in MOUSE], 'alpha': FP_ALPHA, 'sets': FP_IDS})
C_SETS = pd.DataFrame({'set': FP_IDS, 'name': [short(s) for s in FP_IDS], 'members': [len(SET_FP[s]) for s in FP_IDS],
                       'I': C_RES['sets|I'], 'I within': C_RES['sets|I within'],
                       'median abs error': C_RES['sets|median abs error']})
rand = pd.DataFrame({'size': C_RES['random|size'], 'I': C_RES['random|I'], 'I within': C_RES['random|I within']})
q95 = rand.groupby('size').I.quantile(.95)
C_SETS['random 95th (interpolated)'] = np.interp(np.log(C_SETS.members), np.log(q95.index.to_numpy(float)), q95.to_numpy())
C_SETS['above random'] = C_SETS.I > C_SETS['random 95th (interpolated)']
I_LAB, I_ALL = float(C_RES['labels|I']), float(C_RES['all|I'])
C_SETS['above labels'] = C_SETS.I > I_LAB
C_SETS = C_SETS.sort_values('I', ascending=False)
C_SETS.to_csv(tables / 'c_set_information.csv', index=False)
print(f"C: labels {I_LAB:.3f} bits (median error {float(C_RES['labels|median abs error']):.3f}); all fold-protected genes "
      f"{I_ALL:.3f} bits, within labels {float(C_RES['all|I within']):.3f} (error {float(C_RES['all|median abs error']):.3f}); "
      f"labels + genes {float(C_RES['all+labels|I']):.3f}")
print(f"C: {int(C_SETS['above labels'].sum())} of {len(C_SETS)} sets carry more information than the labels; "
      f"{int(C_SETS['above random'].sum())} above their size's random 95th")
display(C_SETS.head(25).round(3))

TOP_C = C_SETS.drop_duplicates('name').head(3)
onoff_rows = []
for s in TOP_C.set:
    a = info_both(set_cols(s), onoff=True)
    onoff_rows.append({'set': short(s), 'counts I': float(C_SETS.set_index('set').loc[s, 'I']), 'on/off I': float(a['I'])})
a = info_both(np.arange(int(FP.sum())), onoff=True)
onoff_rows.append({'set': 'all fold-protected genes', 'counts I': I_ALL, 'on/off I': float(a['I'])})
C_ONOFF = pd.DataFrame(onoff_rows)
C_ONOFF.to_csv(tables / 'c_on_off.csv', index=False)
display(C_ONOFF.round(3))

### Figure 90c

In [ ]:
fig = plt.figure(figsize=(180 * MM, 100 * MM))
gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1, 1], hspace=.5, wspace=.45)
ax = fig.add_subplot(gs[:, 0])
qs = rand.groupby('size').I.quantile([.05, .5, .95]).unstack()
ax.fill_between(qs.index, qs[.05], qs[.95], color=LIGHT, lw=0)
ax.plot(qs.index, qs[.5], color=GREY, lw=.8)
ax.scatter(C_SETS.members, C_SETS.I, s=3, color=INK, alpha=.45, lw=0, rasterized=True)
top3 = C_SETS.drop_duplicates('name').head(3)
ax.scatter(top3.members, top3.I, s=12, color='#D55E00', lw=0)
ax.axhline(I_LAB, color='#4C72B0', lw=1, ls='--')
ax.axhline(I_ALL, color=INK, lw=1)
ax.set_xscale('log')
ax.set_xlabel('fold-protected genes in the set')
ax.set_ylabel('positional information (bits)')
ax.legend(handles=[plt.Line2D([], [], color=INK, label='all fold-protected genes'),
                   plt.Line2D([], [], color='#4C72B0', ls='--', label='S1/S2/S3 labels'),
                   plt.Rectangle((0, 0), 1, 1, color=LIGHT, label='random genes, 5–95%'),
                   plt.Line2D([], [], marker='o', ls='', ms=3, color='#D55E00', label='top sets (panel b)')], fontsize=5,
          loc='lower right', frameon=True, facecolor='white', edgecolor='none', framealpha=.9)
panel(ax, 'a')

ax = fig.add_subplot(gs[0, 1])
bars = [('labels', I_LAB, 0.)] + [(r['name'][:24], r.I, r['I within']) for _, r in top3.iterrows()] + \
       [('all genes', I_ALL, float(C_RES['all|I within']))]
x = np.arange(len(bars))
ax.bar(x - .2, [b[1] for b in bars], width=.38, color=INK)
ax.bar(x + .2, [b[2] for b in bars], width=.38, color='#D55E00')
ax.set_xticks(x, [b[0] for b in bars], rotation=35, ha='right', fontsize=5)
ax.set_ylabel('bits')
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=INK, label='total'),
                   plt.Rectangle((0, 0), 1, 1, color='#D55E00', label='within segments')], fontsize=5,
          loc='lower center', bbox_to_anchor=(.5, 1.0), ncol=2)
panel(ax, 'b')

ax = fig.add_subplot(gs[1, 1])
ax.plot(CENTRES, C_RES['labels|rmse by bin'], color='#4C72B0', ls='--')
ax.plot(CENTRES, C_RES['all|rmse by bin'], color=INK)
sets_idx = {s: i for i, s in enumerate(FP_IDS)}
for s in top3.set:
    ax.plot(CENTRES, C_RES['sets|rmse by bin'][sets_idx[s]], color='#D55E00', lw=.7, alpha=.8)
seg_lines(ax)
ax.set_xlabel('mouse segment units')
ax.set_ylabel('position error (units)')
ax.legend(handles=[plt.Line2D([], [], color='#4C72B0', ls='--', label='labels'), plt.Line2D([], [], color=INK, label='all genes'),
                   plt.Line2D([], [], color='#D55E00', lw=.7, label='top sets')], fontsize=4.5, loc='upper right')
panel(ax, 'c')

ax = fig.add_subplot(gs[:, 2])
s0 = top3.set.iloc[0]
post = decode(set_cols(s0), MOUSE[0], MOUSE[1])
mean_pos = np.exp(post) @ CENTRES
te = DEC[MOUSE[1]]
ax.scatter(te['units'], mean_pos, s=2, c=[LABEL_COLOR[LABELS[k]] for k in te['label']], lw=0, alpha=.6, rasterized=True)
ax.plot([0, 2.5], [0, 2.5], color=INK, lw=.4, ls=':')
seg_lines(ax)
seg_lines(ax, vertical=False)
ax.set_xlabel(f'curve position, {MOUSE[1]}')
ax.set_ylabel('decoded position')
ax.set_xlim(0, 2.5)
ax.set_ylim(0, 2.5)
panel(ax, 'd')
save(fig, 'fig90c_positional_information')

## 7 · D · Human against mouse without levels, against within-species baselines

In [ ]:
pairs = []
for g in GENES71:
    if g in m['index'].index and TO_HUMAN.get(g) in hh['index'].index:
        i, j = int(m['index'][g]), int(hh['index'][TO_HUMAN[g]])
        if m['usable'][i] and hh['usable'][j]:
            pairs.append((g, TO_HUMAN[g], i, j))
D_PAIRS = pd.DataFrame(pairs, columns=['mouse', 'human', 'i', 'j'])
E = {}
RHO = {}
for n in MOUSE:
    E[n] = ca.end_contrast(m['deviation'][n][D_PAIRS.i.to_numpy()], N_END)
    RHO[n] = np.array([spearmanr(np.arange(NB), v, nan_policy='omit')[0] for v in m['deviation'][n][D_PAIRS.i.to_numpy()]])
for n in HUMAN:
    E[n] = ca.end_contrast(hh['deviation'][n][D_PAIRS.j.to_numpy()], N_END)
    RHO[n] = np.array([spearmanr(np.arange(25), v, nan_policy='omit')[0] for v in hh['deviation'][n][D_PAIRS.j.to_numpy()]])
D_TAB = D_PAIRS[['mouse', 'human']].copy()
for n in MOUSE + HUMAN:
    D_TAB[f'E {n}'] = E[n]
    D_TAB[f'rho {n}'] = RHO[n]
D_TAB['E mouse'] = (E[MOUSE[0]] + E[MOUSE[1]]) / 2
D_TAB['E human'] = (E[HUMAN[0]] + E[HUMAN[1]]) / 2
D_TAB['fold-protected'] = ~D_TAB.mouse.isin(BUILT)
D_TAB['category'] = ca.graded_categories(E[MOUSE[0]], E[MOUSE[1]], E[HUMAN[0]], E[HUMAN[1]], graded=GRADED, flat=FLATD)
D_TAB['category'] = D_TAB['category'].replace({'first only': 'mouse only', 'second only': 'human only'})
RELABEL = {'A2+COR1 | A4+MED1': (MOUSE[0], HUMAN[0], MOUSE[1], HUMAN[1]),
           'A2+MED1 | A4+COR1': (MOUSE[0], HUMAN[1], MOUSE[1], HUMAN[0])}


def agreement(mask):
    r = lambda a, b: float(np.corrcoef(a[mask], b[mask])[0, 1])
    out = {'mouse A2 vs A4': r(E[MOUSE[0]], E[MOUSE[1]]), 'human COR1 vs MED1': r(E[HUMAN[0]], E[HUMAN[1]])}
    for a in MOUSE:
        for b in HUMAN:
            out[f'{a} vs {b}'] = r(E[a], E[b])
    out['human vs mouse (means)'] = r(D_TAB['E human'].to_numpy(), D_TAB['E mouse'].to_numpy())
    out['corrected conservation'] = ca.disattenuated_r(out['human vs mouse (means)'], out['mouse A2 vs A4'],
                                                       out['human COR1 vs MED1'])
    return out


D_AGREE = pd.DataFrame({'all': agreement(np.ones(len(D_TAB), bool)), 'fold-protected': agreement(D_TAB['fold-protected'].to_numpy())})
display(D_AGREE.round(3))
cat_counts = {'species': D_TAB['category'].value_counts()}
for k, (a1, a2, b1, b2) in RELABEL.items():
    c = pd.Series(ca.graded_categories(E[a1], E[a2], E[b1], E[b2], graded=GRADED, flat=FLATD))
    cat_counts[k] = c.replace({'first only': 'mouse only', 'second only': 'human only'}).value_counts()
D_COUNTS = pd.DataFrame(cat_counts).reindex(['conserved', 'reversed', 'mouse only', 'human only']).fillna(0).astype(int)
display(D_COUNTS)
D_TAB['strength'] = np.min(np.abs(np.column_stack([E[n] for n in MOUSE + HUMAN])), axis=1)
D_TAB.to_csv(tables / 'd_end_contrasts.csv.gz', index=False)
for cat in ('conserved', 'reversed', 'mouse only', 'human only'):
    sub_ = D_TAB[D_TAB.category.eq(cat)]
    key = sub_['strength'] if cat in ('conserved', 'reversed') else (sub_['E mouse'].abs() if cat == 'mouse only' else sub_['E human'].abs())
    print(cat, ', '.join(sub_.loc[key.sort_values(ascending=False).index[:15], 'mouse']))

### Figure 90d

In [ ]:
CAT_COLOR = {'conserved': '#009E73', 'reversed': '#D55E00', 'mouse only': '#0072B2', 'human only': '#E69F00'}
fig = plt.figure(figsize=(180 * MM, 118 * MM))
gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1.35, .9], hspace=.55, wspace=.75)
lim = 4
for k, (a, b) in enumerate(((MOUSE[0], MOUSE[1]), (HUMAN[0], HUMAN[1]))):
    ax = fig.add_subplot(gs[0, k])
    ax.hexbin(np.clip(E[a], -lim, lim), np.clip(E[b], -lim, lim), gridsize=40, bins='log', cmap='Greys', mincnt=1,
              linewidths=0, rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], color=INK, lw=.4, ls=':')
    ax.set_xlabel(f'end contrast, {a}')
    ax.set_ylabel(f'end contrast, {b}')
    ax.set_aspect('equal')
    panel(ax, 'ab'[k])
ax = fig.add_subplot(gs[0, 2])
ax.hexbin(np.clip(D_TAB['E mouse'], -lim, lim), np.clip(D_TAB['E human'], -lim, lim), gridsize=40, bins='log', cmap='Greys',
          mincnt=1, linewidths=0, rasterized=True)
for cat, col in CAT_COLOR.items():
    sub_ = D_TAB[D_TAB.category.eq(cat)]
    ax.scatter(sub_['E mouse'].clip(-lim, lim), sub_['E human'].clip(-lim, lim), s=3, color=col, lw=0, label=cat)
    key = sub_['strength'] if cat in ('conserved', 'reversed') else (sub_['E mouse'].abs() if cat == 'mouse only' else sub_['E human'].abs())
    for j_, (_, r) in enumerate(sub_.loc[key.sort_values(ascending=False).index[:2]].iterrows()):
        ax.annotate(r['mouse'], (np.clip(r['E mouse'], -lim, lim), np.clip(r['E human'], -lim, lim)), fontsize=4.8,
                    style='italic', xytext=(3, 6 - 10 * j_), textcoords='offset points', color=col)
ax.axhline(0, color=LIGHT, lw=.5)
ax.axvline(0, color=LIGHT, lw=.5)
ax.set_xlabel('end contrast, mouse')
ax.set_ylabel('end contrast, human')
ax.set_aspect('equal')
ax.legend(fontsize=5, loc='upper left', bbox_to_anchor=(-.05, -.3), ncol=2, markerscale=2)
panel(ax, 'c')

ax = fig.add_subplot(gs[0, 3])
keys = ['mouse A2 vs A4', 'human COR1 vs MED1'] + [f'{a} vs {b}' for a in MOUSE for b in HUMAN] + ['corrected conservation']
vals = D_AGREE.loc[keys, 'all']
cols = ['#0072B2', '#E69F00'] + [GREY] * 4 + [INK]
ax.barh(np.arange(len(keys))[::-1], vals, color=cols)
ax.set_yticks(np.arange(len(keys))[::-1], ['A2–A4', 'COR1–MED1', 'A2–COR1', 'A2–MED1', 'A4–COR1', 'A4–MED1', 'corrected'],
              fontsize=5.5)
ax.set_xlabel('r (end contrast)')
ax.set_xlim(0, 1)
panel(ax, 'd')

ax = fig.add_subplot(gs[1, 0])
x = np.arange(4)
for k, (col_name, col) in enumerate((('species', INK), ('A2+COR1 | A4+MED1', GREY), ('A2+MED1 | A4+COR1', LIGHT))):
    ax.bar(x + (k - 1) * .27, D_COUNTS[col_name], width=.26, color=col)
ax.set_xticks(x, D_COUNTS.index, rotation=30, ha='right')
ax.set_ylabel('genes')
ax.set_yscale('symlog', linthresh=10)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c_, label=l_) for c_, l_ in
                   ((INK, 'species'), (GREY, 'A2+COR1 | A4+MED1'), (LIGHT, 'A2+MED1 | A4+COR1'))], fontsize=4.5,
          loc='upper right', bbox_to_anchor=(1.15, 1.12))
panel(ax, 'e')

sub = gs[1, 1:].subgridspec(2, 4, hspace=.6, wspace=.35)
hx = {n: np.linspace(0, 1, 25) for n in HUMAN}
ex_genes = []
for cat in ('conserved', 'reversed', 'mouse only', 'human only'):
    sub_ = D_TAB[D_TAB.category.eq(cat)]
    key = sub_['strength'] if cat in ('conserved', 'reversed') else (sub_['E mouse'].abs() if cat == 'mouse only' else sub_['E human'].abs())
    ex_genes += [(cat, i) for i in key.sort_values(ascending=False).index[:2]]
for k, (cat, i) in enumerate(ex_genes[:8]):
    ax = fig.add_subplot(sub[k // 4, k % 4])
    r = D_TAB.loc[i]
    for n in MOUSE:
        ax.plot(CENTRES / 2.5, m['deviation'][n][D_PAIRS.loc[i, 'i']], color=SPEC_COLOR[n], lw=.8)
    for n in HUMAN:
        ax.plot(hx[n], hh['deviation'][n][D_PAIRS.loc[i, 'j']], color=SPEC_COLOR[n], lw=.8)
    ax.set_title(r['mouse'], fontsize=5.5, style='italic', fontweight='normal', color=CAT_COLOR[cat])
    ax.tick_params(labelsize=4.5)
    if k == 0:
        panel(ax, 'f')
        ax.legend(handles=[plt.Line2D([], [], color=SPEC_COLOR[n], label=n) for n in MOUSE + HUMAN], fontsize=4,
                  loc='upper left', handlelength=1, borderpad=.1, labelspacing=.2)
save(fig, 'fig90d_species_level_free')

## 8 · E · Where along the PT the transcriptome changes

In [ ]:
LC = {n: ca.local_change(DM[n], HALF_LC) for n in MOUSE}
LC_POS = np.arange(HALF_LC, NB - HALF_LC + 1) * .1


def agreed(a, b, thr=EFFECT):
    up = (a >= thr) & (b >= thr)
    down = (a <= -thr) & (b <= -thr)
    return up, down


UP, DOWN = agreed(LC[MOUSE[0]], LC[MOUSE[1]])


def chance_compute():
    out_up, out_down = [], []
    for p in PERMS[:N_PERM_LC]:
        u, d = agreed(LC[MOUSE[0]], LC[MOUSE[1]][p])
        out_up.append(u.sum(axis=0))
        out_down.append(d.sum(axis=0))
    return {'up': np.array(out_up), 'down': np.array(out_down)}


CHANCE = cached('e_chance', chance_compute, params={'n': N_PERM_LC, 'seed': SEED}, inputs={'lc': [LC[n] for n in MOUSE]})
DISC = ca.discrete_profiles(m['counts'], m['library'], m['segment'], BIN, NB, m['specimen'], MOUSE, LABELS)
DISC_DEV = {n: DISC[n][U_IDX] - pp.reference_profile(DISC[n], m['usable']) for n in MOUSE}
LC_D = {n: ca.local_change(DISC_DEV[n], HALF_LC) for n in MOUSE}
UP_D, DOWN_D = agreed(LC_D[MOUSE[0]], LC_D[MOUSE[1]])
E_TRACK = pd.DataFrame({'position': LC_POS, 'rising': UP.sum(axis=0), 'falling': DOWN.sum(axis=0),
                        'rising chance': CHANCE['up'].mean(axis=0), 'falling chance': CHANCE['down'].mean(axis=0),
                        'rising chance 95': np.quantile(CHANCE['up'], .95, axis=0),
                        'falling chance 95': np.quantile(CHANCE['down'], .95, axis=0),
                        'rising discrete': UP_D.sum(axis=0), 'falling discrete': DOWN_D.sum(axis=0),
                        'rising fold-protected': UP[FP].sum(axis=0), 'falling fold-protected': DOWN[FP].sum(axis=0)})
E_TRACK['excess'] = (E_TRACK.rising + E_TRACK.falling - E_TRACK['rising chance'] - E_TRACK['falling chance']).clip(lower=0)
E_TRACK['discrete total'] = E_TRACK['rising discrete'] + E_TRACK['falling discrete']
inside = (np.abs(LC_POS - 1) > .3) & (np.abs(LC_POS - 2) > .3)
E_SHARE = pd.Series({'observed excess inside segments': float(E_TRACK.loc[inside, 'excess'].sum() / max(E_TRACK['excess'].sum(), 1e-9)),
                     'discrete inside segments': float(E_TRACK.loc[inside, 'discrete total'].sum() / max(E_TRACK['discrete total'].sum(), 1e-9)),
                     'positions inside segments': float(inside.mean())})
E_TRACK.to_csv(tables / 'e_change_density.csv', index=False)
display(E_TRACK.round(1))
display(E_SHARE.round(3))

# Post hoc (descriptive): reproducible change energy. The product of the two mice's local changes has expectation
# equal to the product of the true changes, whatever each bin's noise, so it needs no threshold and no chance
# subtraction, and bin occupancy does not bias it.
READING.append('E post hoc: reproducible change energy Σ_g Δ_A·Δ_B per position (and per set, as a member mean), '
               'because the frozen agreed-change counts are dominated by chance agreement (chance ≈ 85–95% of the '
               'observed counts) and their power depends on bin occupancy.')
ENERGY = LC[MOUSE[0]] * LC[MOUSE[1]]
ENERGY_D = LC_D[MOUSE[0]] * LC_D[MOUSE[1]]
E_TRACK['energy'] = ENERGY.sum(axis=0)
E_TRACK['energy fold-protected'] = ENERGY[FP].sum(axis=0)
E_TRACK['energy steps'] = ENERGY_D.sum(axis=0)
pos_e = E_TRACK['energy'].clip(lower=0)
E_SHARE['energy inside segments (post hoc)'] = float(pos_e[inside].sum() / pos_e.sum())
E_SHARE['energy inside segments, fold-protected (post hoc)'] = float(E_TRACK['energy fold-protected'].clip(lower=0)[inside].sum()
                                                                    / E_TRACK['energy fold-protected'].clip(lower=0).sum())
E_SHARE['steps energy inside segments (post hoc)'] = float(E_TRACK['energy steps'].clip(lower=0)[inside].sum()
                                                           / E_TRACK['energy steps'].clip(lower=0).sum())
E_SHARE['energy captured by steps (post hoc)'] = float(E_TRACK['energy steps'].sum() / E_TRACK['energy'].sum())
display(E_SHARE.round(3))
BG_ENERGY = ENERGY.mean(axis=0)
SET_ENERGY = pd.DataFrame(np.vstack([ENERGY[SET_U[s]].mean(axis=0) - BG_ENERGY for s in SET_IDS]), index=SET_IDS,
                          columns=np.round(LC_POS, 1))
SET_ENERGY.to_csv(tables / 'e_set_change_energy_post_hoc.csv')
E_TRACK.to_csv(tables / 'e_change_density.csv', index=False)

WIN_EDGES = np.arange(.2, 2.31 + 1e-9, WINDOW)
WIN = np.clip(np.digitize(LC_POS, WIN_EDGES) - 1, 0, len(WIN_EDGES) - 2)
E_ROWS = []
universe = len(U_IDX)
for w in range(len(WIN_EDGES) - 1):
    cols = WIN == w
    for direction, mask in (('rising', UP), ('falling', DOWN)):
        hit = mask[:, cols].any(axis=1)
        k_hit = int(hit.sum())
        for s in SET_IDS:
            u = SET_U[s]
            a = int(hit[u].sum())
            table = [[a, len(u) - a], [k_hit - a, universe - len(u) - k_hit + a]]
            E_ROWS.append({'window': f'{WIN_EDGES[w]:.1f}–{WIN_EDGES[w + 1]:.1f}', 'w': w, 'direction': direction, 'set': s,
                           'hits': a, 'members': len(u), 'genes changing': k_hit,
                           'p': fisher_exact(table, alternative='greater')[1]})
E_ENR = pd.DataFrame(E_ROWS)
E_ENR['q'] = E_ENR.groupby(['w', 'direction']).p.transform(lambda p: multipletests(p, method='fdr_bh')[1])
E_ENR['name'] = [short(s) for s in E_ENR.set]
E_ENR.to_csv(tables / 'e_window_enrichment.csv.gz', index=False)
sig = E_ENR[E_ENR.q <= .05]
print(f'E: {sig.set.nunique()} sets enriched in at least one window')
display(sig.sort_values('q').groupby(['w', 'direction']).head(4)[['window', 'direction', 'name', 'hits', 'members', 'q']])

### Figure 90e

In [ ]:
fig = plt.figure(figsize=(180 * MM, 112 * MM))
gs = fig.add_gridspec(2, 3, width_ratios=[1.45, 1.45, .55], height_ratios=[1, 1.35], hspace=.5, wspace=.5)
ax = fig.add_subplot(gs[0, 0])
ax.plot(LC_POS, E_TRACK.rising - E_TRACK['rising chance'], color='#C44E52', lw=1.1, drawstyle='steps-mid')
ax.plot(LC_POS, -(E_TRACK.falling - E_TRACK['falling chance']), color='#4C72B0', lw=1.1, drawstyle='steps-mid')
ax.plot(LC_POS, E_TRACK['rising discrete'], color='#C44E52', lw=.8, ls='--', drawstyle='steps-mid')
ax.plot(LC_POS, -E_TRACK['falling discrete'], color='#4C72B0', lw=.8, ls='--', drawstyle='steps-mid')
ax.axhline(0, color=INK, lw=.4)
seg_lines(ax)
ax.set_xlim(0, 2.5)
ax.set_xlabel('mouse segment units')
ax.set_ylabel('genes changing beyond chance')
ax.legend(handles=[plt.Line2D([], [], color='#C44E52', label='rising'), plt.Line2D([], [], color='#4C72B0', label='falling'),
                   plt.Line2D([], [], color=GREY, ls='--', label='steps at labels')], fontsize=5, loc='lower left')
panel(ax, 'a')

ax = fig.add_subplot(gs[0, 1])
tot = E_TRACK['energy'].clip(lower=0)
ax.fill_between(LC_POS, 0, tot / tot.sum(), color=INK, alpha=.25, step='mid', lw=0)
ax.plot(LC_POS, tot / tot.sum(), color=INK, lw=1.1, drawstyle='steps-mid')
fpe = E_TRACK['energy fold-protected'].clip(lower=0)
ax.plot(LC_POS, fpe / fpe.sum(), color='#D55E00', lw=.8, drawstyle='steps-mid')
de = E_TRACK['energy steps'].clip(lower=0)
ax.plot(LC_POS, de / de.sum(), color=GREY, lw=1, ls='--', drawstyle='steps-mid')
seg_lines(ax)
ax.set_xlim(0, 2.5)
ax.set_xlabel('mouse segment units')
ax.set_ylabel('share of reproducible change')
ax.legend(handles=[plt.Line2D([], [], color=INK, label='all genes'), plt.Line2D([], [], color='#D55E00', label='fold-protected'),
                   plt.Line2D([], [], color=GREY, ls='--', label='steps at labels')], fontsize=5, loc='upper left')
panel(ax, 'b')

ax = fig.add_subplot(gs[0, 2])
vals = [E_SHARE['energy inside segments (post hoc)'], E_SHARE['steps energy inside segments (post hoc)']]
ax.bar([0, 1], vals, color=[INK, GREY])
ax.axhline(E_SHARE['positions inside segments'], color=INK, lw=.5, ls=':')
ax.set_xticks([0, 1], ['observed', 'steps'])
ax.set_ylabel('share of change inside segments')
ax.set_ylim(0, 1)
panel(ax, 'c')

ax = fig.add_subplot(gs[1, :])
cand = SET_ENERGY.clip(lower=0).sum(axis=1).sort_values(ascending=False)
grp = pp.redundancy_groups({s: [U_GENES[i] for i in SET_U[s]] for s in cand.index[:150]})
chosen = []
for s in cand.index[:150]:
    if grp[s] not in [grp[c] for c in chosen]:
        chosen.append(s)
    if len(chosen) == 32:
        break
Mx = SET_ENERGY.loc[chosen].clip(lower=0)
Mx = Mx.div(Mx.max(axis=1), axis=0)
Mx = Mx.iloc[np.argsort([np.average(LC_POS, weights=r ** 2 + 1e-12) for r in Mx.to_numpy()])]
im = ax.imshow(Mx.to_numpy().T, aspect='auto', cmap='Greys', vmin=0, vmax=1, interpolation='nearest',
               extent=(-.5, len(Mx) - .5, LC_POS[-1] + .05, LC_POS[0] - .05))
ax.set_xticks(np.arange(len(Mx)), [short(s, 34) for s in Mx.index], rotation=55, ha='right', fontsize=4.8)
ax.set_ylabel('mouse segment units')
seg_lines(ax, vertical=False)
for b in (1, 2):
    ax.axhline(b, color='#D55E00', lw=.6, ls=':')
cb = fig.colorbar(im, ax=ax, fraction=.02, pad=.01)
cb.set_label('reproducible change (row max = 1)')
panel(ax, 'd')
save(fig, 'fig90e_change_density')

## 9 · F · Where along the PT injury acts

89's E4 inputs: AKI export, U_PT units of the four mice on the control mouse units, PT-expressed genes. Δ(x) = IR
mean − control mean of binned log2 rates, centred by the median usable gene.

In [ ]:
AKI = read_pt_units(EXPORT['aki'], 'aki')
assert AKI.uns['provenance']['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT['aki']
aobs = AKI.obs
a_spec_all = aobs['sample'].astype(str).to_numpy()
a_seg_all = aobs.segment_class.astype(str).to_numpy()
a_mask, a_pos = support_positions(aobs.pseudospace.to_numpy(float), a_spec_all, a_seg_all)
a_spec, a_seg = a_spec_all[a_mask], a_seg_all[a_mask]
a_ident = aobs.primary_identity.astype(str).to_numpy()[a_mask]
a_tr = logistic_transitions(a_pos, a_seg, a_spec)
for n in CONTROLS:
    t = a_tr[a_tr.specimen.eq(n)].sort_values('transition').position.to_numpy()
    assert len(t) == 2 and t[0] < t[1], f'G7: {n} transitions out of order'
AKI_CUTS = sc.mean_transitions(a_tr, CONTROLS)
a_units = sc.to_segment_units(a_pos, AKI_CUTS)
GUARDS.append(f'G7: AKI control transitions ordered; cuts {AKI_CUTS[0]:.3f}, {AKI_CUTS[1]:.3f}')
a_pt = a_ident == 'Proximal_Tubule'
a_x = sparse.csr_matrix(AKI.X, dtype=float)[np.flatnonzero(a_mask)][a_pt]
a_genes_all = pd.Index(AKI.var_names.astype(str))
a_not_mt = ~a_genes_all.str.startswith('mt-')
a_lib = np.asarray(a_x[:, np.flatnonzero(a_not_mt)].sum(axis=1)).ravel()
a_spec, a_units, a_seg = a_spec[a_pt], a_units[a_pt], a_seg[a_pt]
a_det = detection_by(a_x, a_spec, AKI4)
a_keep = (a_det >= DETECTION).all(axis=0) & a_not_mt
A_GENES = np.array(a_genes_all[a_keep])
a_x = a_x[:, np.flatnonzero(a_keep)]
A_USABLE = (a_det[:, a_keep] >= USABLE).all(axis=0)
a_bin, _ = pp.fixed_bins(a_units)
A_RATES = pp.bin_profiles(a_x, a_lib, a_bin, NB, a_spec, AKI4, centre=False)
a_valid = np.all([np.isfinite(A_RATES[n][0]) for n in AKI4], axis=0)
print(f'AKI: {len(a_spec):,} U_PT units {pd.Series(a_spec).value_counts().to_dict()}; {len(A_GENES):,} genes '
      f'({int(A_USABLE.sum()):,} usable); valid bins {int(a_valid.sum())}/{NB}')


def a_delta(case, ref):
    d = np.mean([A_RATES[n] for n in case], axis=0) - np.mean([A_RATES[n] for n in ref], axis=0)
    return pp.centre_by_median(d, A_USABLE)


DELTA = {'condition': a_delta(IRS, CONTROLS),
         'relabel A2|A4': a_delta(['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
         'relabel cross': a_delta(['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2'])}
PAIRS = [(i, c) for i in IRS for c in CONTROLS]
for i, c in PAIRS:
    DELTA[f'{i}-{c}'] = a_delta([i], [c])
dc = DELTA['condition']
pair_sign = np.all([np.sign(DELTA[f'{i}-{c}']) == np.sign(dc) for i, c in PAIRS], axis=0)
relab = np.maximum(np.abs(DELTA['relabel A2|A4']), np.abs(DELTA['relabel cross']))
resp_bin = (np.abs(dc) >= RESP) & pair_sign & (relab <= RESP_RELABEL * np.abs(dc)) & a_valid[None, :]
adjacent = (resp_bin[:, 1:] & resp_bin[:, :-1]).any(axis=1)
RESPONSIVE = np.flatnonzero(adjacent & A_USABLE)
F_ROWS = []
for g in RESPONSIVE:
    rb = resp_bin[g]
    j = int(np.argmax(np.where(rb, np.abs(dc[g]), -1)))
    v = a_valid
    F_ROWS.append({'gene': A_GENES[g], 'direction': 'up' if dc[g, j] > 0 else 'down', 'peak': float(CENTRES[j]),
                   'peak Δ': float(dc[g, j]), 'bins responding': int(rb.sum()),
                   'first responding': float(CENTRES[np.flatnonzero(rb)[0]]),
                   'effect gradient (Spearman |Δ| vs position)': float(spearmanr(CENTRES[v], np.abs(dc[g, v]))[0])})
F_TAB = pd.DataFrame(F_ROWS)


def responsive_genes(case, ref):
    """The protocol's responsive rule with ``case`` vs ``ref`` as the contrast; the other two balanced splits of the
    four mice are its relabelings."""
    d = a_delta(case, ref)
    sign_ok = np.all([np.sign(a_delta([i], [c])) == np.sign(d) for i in case for c in ref], axis=0)
    others = [(a, b) for a, b in ((['IR2A2', 'IR2A4'], ['Ctrl1A2', 'Ctrl1A4']), (['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4']),
                                  (['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2']))
              if sorted(a) != sorted(case) and sorted(a) != sorted(ref)]
    rl = np.max([np.abs(a_delta(a, b)) for a, b in others], axis=0)
    rb = (np.abs(d) >= RESP) & sign_ok & (rl <= RESP_RELABEL * np.abs(d)) & a_valid[None, :]
    return np.flatnonzero((rb[:, 1:] & rb[:, :-1]).any(axis=1) & A_USABLE)


READING.append('F post hoc: the responsive rule applied to the two relabelings as contrasts (each with the other two '
               'splits as its relabelings), as the specificity check the protocol did not state.')
F_SPEC = pd.Series({'IR vs control': len(RESPONSIVE),
                    'A2 mice vs A4 mice': len(responsive_genes(['IR2A2', 'Ctrl1A2'], ['IR2A4', 'Ctrl1A4'])),
                    'IR2A2+Ctrl1A4 vs IR2A4+Ctrl1A2': len(responsive_genes(['IR2A2', 'Ctrl1A4'], ['IR2A4', 'Ctrl1A2']))})
display(F_SPEC)
F_TAB['peak segment'] = pd.cut(F_TAB.peak, [0, 1, 2, 2.5], labels=SEGMENTS, include_lowest=True).astype(str)
F_TAB.to_csv(tables / 'f_responsive_genes.csv', index=False)
print(f'F: {len(F_TAB)} responsive genes ({F_TAB.direction.value_counts().to_dict()})')
display(pd.crosstab(F_TAB.direction, F_TAB['peak segment']))
display(F_TAB.groupby('direction')['effect gradient (Spearman |Δ| vs position)'].describe().round(3))

cond = np.where(np.isin(a_spec, IRS), 'IR', 'control')
MATCH = []
for lab in LABELS:
    pooled, matched, shift = ca.matched_fold_changes(a_x, a_lib, a_bin, a_seg, cond, NB, lab, case='IR', ref='control')
    MATCH.append((lab, pooled, matched, shift * .1))
F_MATCH = pd.DataFrame([{'label': lab, 'position shift (units)': sh,
                         'segment |FC| ≥ 0.5, matched < 0.25': int(((np.abs(p) >= RESP) & (np.abs(q) < EFFECT) & A_USABLE).sum()),
                         'matched |FC| ≥ 0.5, segment < 0.25': int(((np.abs(q) >= RESP) & (np.abs(p) < EFFECT) & A_USABLE).sum()),
                         'r(segment, matched)': float(np.corrcoef(p[A_USABLE & np.isfinite(q)], q[A_USABLE & np.isfinite(q)])[0, 1])}
                        for lab, p, q, sh in MATCH])
display(F_MATCH.round(3))
DENS = {n: np.bincount(a_bin[(a_spec == n) & (a_bin >= 0)], minlength=NB) / ((a_spec == n) & (a_bin >= 0)).sum() for n in AKI4}

### Figure 90f

In [ ]:
fig = plt.figure(figsize=(180 * MM, 112 * MM))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1.25, 1], hspace=.55, wspace=.6)
ax = fig.add_subplot(gs[0, 0])
for n in AKI4:
    ax.plot(CENTRES, DENS[n], color=SPEC_COLOR[n], lw=1)
seg_lines(ax)
ax.set_xlabel('mouse segment units')
ax.set_ylabel('share of PT units')
ax.legend(fontsize=4.5, labels=AKI4, handlelength=1, loc='upper right', borderpad=.2)
panel(ax, 'a')

ax = fig.add_subplot(gs[:, 1])
ft = F_TAB.copy()
ft['g'] = [int(np.flatnonzero(A_GENES == g)[0]) for g in ft.gene]
ft = ft.sort_values(['direction', 'peak'], ascending=[False, True])
im = ax.imshow(np.clip(np.where(a_valid, dc[ft.g], np.nan), -2, 2), aspect='auto', cmap='RdBu_r', norm=TwoSlopeNorm(0, -2, 2),
               extent=(0, 2.5, len(ft), 0), interpolation='nearest', rasterized=True)
n_up = int(ft.direction.eq('up').sum())
ax.axhline(n_up, color=INK, lw=.5)
seg_lines(ax)
ax.set_yticks([])
ax.set_xlabel('mouse segment units')
ax.set_ylabel('responsive genes')
cb = fig.colorbar(im, ax=ax, fraction=.04, pad=.02, location='top')
cb.set_label('IR − control (log2, centred)')
panel(ax, 'b')

ax = fig.add_subplot(gs[1, 0])
for direction, col in (('up', '#C44E52'), ('down', '#4C72B0')):
    gi = ft.loc[ft.direction.eq(direction), 'g'].to_numpy()
    if not len(gi):
        continue
    ax.plot(CENTRES[a_valid], np.median(dc[gi], axis=0)[a_valid], color=col, lw=1.2)
    for key_ in ('relabel A2|A4', 'relabel cross'):
        ax.plot(CENTRES[a_valid], np.median(DELTA[key_][gi], axis=0)[a_valid], color=col, lw=.6, ls=':')
ax.axhline(0, color=INK, lw=.4)
seg_lines(ax)
ax.set_xlabel('mouse segment units')
ax.set_ylabel('median Δ (log2)')
ax.legend(handles=[plt.Line2D([], [], color='#C44E52', label='up genes'), plt.Line2D([], [], color='#4C72B0', label='down genes'),
                   plt.Line2D([], [], color=GREY, ls=':', label='relabelings')], fontsize=4.5, loc='center right')
panel(ax, 'c')

ax = fig.add_subplot(gs[0, 2])
k = int(np.argmax([abs(sh) for _, _, _, sh in MATCH]))
lab, p, q, sh = MATCH[k]
ok = A_USABLE & np.isfinite(q)
ax.scatter(p[ok], q[ok], s=2, color=GREY, lw=0, alpha=.5, rasterized=True)
off = ok & (np.abs(p) >= RESP) & (np.abs(q) < EFFECT)
ax.scatter(p[off], q[off], s=4, color='#D55E00', lw=0)
lim = np.nanquantile(np.abs(np.concatenate([p[ok], q[ok]])), .999)
ax.plot([-lim, lim], [-lim, lim], color=INK, lw=.4, ls=':')
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel(f'{lab} pseudobulk log2 FC')
ax.set_ylabel('position-matched log2 FC')
ax.set_aspect('equal')
panel(ax, 'd')

ax = fig.add_subplot(gs[1, 2])
ax.bar(np.arange(3), F_SPEC.to_numpy(), color=[INK, GREY, LIGHT])
ax.set_xticks(np.arange(3), ['IR vs control', 'A2 vs A4', 'crossed'], rotation=25, ha='right')
ax.set_ylabel('responsive genes')
panel(ax, 'e')
save(fig, 'fig90f_injury_location')

## 10 · Readout and run record

In [ ]:
READOUT = pd.DataFrame([
    {'part': 'A', 'readout': 'classes (all usable)', 'value': json.dumps(A_COUNTS['all usable'].to_dict(), default=int)},
    {'part': 'A', 'readout': 'continuous favoured (fold-protected non-flat); D-truth false continuous; C-truth false discrete',
     'value': f'{real_cont:.3f}; {false_cont:.3f}; {false_disc:.3f}'},
    {'part': 'A', 'readout': 'class agreement between mice (chance)', 'value': f'{agree:.3f} ({agree_chance:.3f})'},
    {'part': 'A', 'readout': 'replicated bumps/troughs', 'value': int(bump_rep.replicated.sum())},
    {'part': 'B', 'readout': 'positioned sets (all; fold-protected)', 'value': f'{int(B_POS.positioned.sum())}; {int(B_POS_FP.positioned.sum())}'},
    {'part': 'B', 'readout': 'hidden by the median (all; fold-protected)',
     'value': f"{int(B_AL['hidden by the median'].sum())}; {int(B_AL_FP['hidden by the median'].sum())}"},
    {'part': 'C', 'readout': 'bits: labels; all fold-protected genes; within labels',
     'value': f"{I_LAB:.3f}; {I_ALL:.3f}; {float(C_RES['all|I within']):.3f}"},
    {'part': 'C', 'readout': 'sets above labels; above random', 'value': f"{int(C_SETS['above labels'].sum())}; {int(C_SETS['above random'].sum())}"},
    {'part': 'D', 'readout': 'r: mouse; human; species means; corrected',
     'value': '; '.join(f'{D_AGREE.loc[k, "all"]:.3f}' for k in ('mouse A2 vs A4', 'human COR1 vs MED1', 'human vs mouse (means)', 'corrected conservation'))},
    {'part': 'D', 'readout': 'categories (species | relabelings)', 'value': json.dumps(D_COUNTS.to_dict(), default=int)},
    {'part': 'E', 'readout': 'share of excess change inside segments (observed; steps)',
     'value': f"{E_SHARE['observed excess inside segments']:.3f}; {E_SHARE['discrete inside segments']:.3f}"},
    {'part': 'F', 'readout': 'responsive genes', 'value': json.dumps(F_TAB.direction.value_counts().to_dict(), default=int)},
    {'part': 'F', 'readout': 'responsive genes, condition and relabelings (post hoc)', 'value': json.dumps(F_SPEC.to_dict(), default=int)},
    {'part': 'E', 'readout': 'reproducible change energy inside segments (post hoc; observed, steps, captured by steps)',
     'value': f"{E_SHARE['energy inside segments (post hoc)']:.3f}; {E_SHARE['steps energy inside segments (post hoc)']:.3f}; {E_SHARE['energy captured by steps (post hoc)']:.3f}"},
    {'part': 'B', 'readout': 'hidden by the median: expected by chance', 'value': f'{HIDDEN_EXPECTED:.1f}'},
    {'part': 'F', 'readout': 'segment-only changes per label', 'value': json.dumps(F_MATCH.set_index('label')['segment |FC| ≥ 0.5, matched < 0.25'].to_dict(), default=int)},
])
READOUT.to_csv(tables / 'readout.csv', index=False)
display(READOUT)
record = {'logic': NOTEBOOK_LOGIC_VERSION, 'guards': GUARDS, 'reading': READING, 'seconds': SECONDS,
          'units': {n: int((m['specimen'] == n).sum()) for n in MOUSE}, 'mouse_cuts': list(MOUSE_CUTS),
          'aki_cuts': list(AKI_CUTS), 'usable_genes': int(len(U_IDX)), 'fold_protected': int(FP.sum()),
          'sets': len(SET_IDS), 'sets_fold_protected': len(FP_IDS), 'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=float))
print(f"Done in {record['wall_seconds'] / 60:.1f} min")